In [11]:
import numpy as np

# List of text documents (corpus)
docs = [
    'go india',
    'india india',
    'hip hip hurray',
    'jeetega bhai jeetega india jeetega',
    'bharat mata ki jai',
    'kohli kohli',
    'sachin sachin',
    'dhoni dhoni',
    'modi ji ki jai',
    'inquilab zindabad'
]

In [15]:
# Import Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer with OOV token
# oov_token means: if any new/unknown word comes later,
# it will be replaced with '<nothing>' instead of being ignored
tokenizer = Tokenizer(oov_token='<nothing>')

In [17]:
# Fit tokenizer on given documents (build vocabulary)
tokenizer.fit_on_texts(docs)

In [31]:
# Dictionary: word → index
tokenizer.word_index

{'<nothing>': 1,
 'india': 2,
 'jeetega': 3,
 'hip': 4,
 'ki': 5,
 'jai': 6,
 'kohli': 7,
 'sachin': 8,
 'dhoni': 9,
 'go': 10,
 'hurray': 11,
 'bhai': 12,
 'bharat': 13,
 'mata': 14,
 'modi': 15,
 'ji': 16,
 'inquilab': 17,
 'zindabad': 18}

In [33]:
# Dictionary: word → frequency count
tokenizer.word_counts

OrderedDict([('go', 1),
             ('india', 4),
             ('hip', 2),
             ('hurray', 1),
             ('jeetega', 3),
             ('bhai', 1),
             ('bharat', 1),
             ('mata', 1),
             ('ki', 2),
             ('jai', 2),
             ('kohli', 2),
             ('sachin', 2),
             ('dhoni', 2),
             ('modi', 1),
             ('ji', 1),
             ('inquilab', 1),
             ('zindabad', 1)])

In [35]:
# Total number of documents
tokenizer.document_count

10

In [37]:
# Convert text into sequences of numbers
sequences = tokenizer.texts_to_sequences(docs)
sequences

[[10, 2],
 [2, 2],
 [4, 4, 11],
 [3, 12, 3, 2, 3],
 [13, 14, 5, 6],
 [7, 7],
 [8, 8],
 [9, 9],
 [15, 16, 5, 6],
 [17, 18]]

# Padding

In [40]:
# Import pad_sequences from TensorFlow Keras (correct way)
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [42]:
# Padding sequences so that all have equal length
# padding='post' → add zeros at the end of each sequence
sequences = pad_sequences(sequences, padding='post')

In [44]:
# Print padded sequences
sequences

array([[10,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 4,  4, 11,  0,  0],
       [ 3, 12,  3,  2,  3],
       [13, 14,  5,  6,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [ 9,  9,  0,  0,  0],
       [15, 16,  5,  6,  0],
       [17, 18,  0,  0,  0]])

## Using Imdb data set 

In [48]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [50]:
# Load IMDB dataset (only top 10,000 words kept)
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)
X_train[0]

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


[1,
 14,
 22,
 16,
 43,
 530,
 973,
 1622,
 1385,
 65,
 458,
 4468,
 66,
 3941,
 4,
 173,
 36,
 256,
 5,
 25,
 100,
 43,
 838,
 112,
 50,
 670,
 2,
 9,
 35,
 480,
 284,
 5,
 150,
 4,
 172,
 112,
 167,
 2,
 336,
 385,
 39,
 4,
 172,
 4536,
 1111,
 17,
 546,
 38,
 13,
 447,
 4,
 192,
 50,
 16,
 6,
 147,
 2025,
 19,
 14,
 22,
 4,
 1920,
 4613,
 469,
 4,
 22,
 71,
 87,
 12,
 16,
 43,
 530,
 38,
 76,
 15,
 13,
 1247,
 4,
 22,
 17,
 515,
 17,
 12,
 16,
 626,
 18,
 2,
 5,
 62,
 386,
 12,
 8,
 316,
 8,
 106,
 5,
 4,
 2223,
 5244,
 16,
 480,
 66,
 3785,
 33,
 4,
 130,
 12,
 16,
 38,
 619,
 5,
 25,
 124,
 51,
 36,
 135,
 48,
 25,
 1415,
 33,
 6,
 22,
 12,
 215,
 28,
 77,
 52,
 5,
 14,
 407,
 16,
 82,
 2,
 8,
 4,
 107,
 117,
 5952,
 15,
 256,
 4,
 2,
 7,
 3766,
 5,
 723,
 36,
 71,
 43,
 530,
 476,
 26,
 400,
 317,
 46,
 7,
 4,
 2,
 1029,
 13,
 104,
 88,
 4,
 381,
 15,
 297,
 98,
 32,
 2071,
 56,
 26,
 141,
 6,
 194,
 7486,
 18,
 4,
 226,
 22,
 21,
 134,
 476,
 26,
 480,
 5,
 144,
 30,
 5535,
 18,

In [51]:
# Pad sequences to fixed length (50 words per review)
# padding='post' → add zeros at end
X_train = pad_sequences(X_train, padding='post', maxlen=50)
X_test = pad_sequences(X_test, padding='post', maxlen=50)

# Build model

In [53]:
# Create a Sequential model (layers will be added one after another)
model = Sequential()

# Add a Simple RNN layer
# 32 → number of neurons (hidden units)
# input_shape=(50,1) → 
#    50 = number of time steps (sequence length)
#    1  = number of features at each time step (each word is treated as a single number)
# return_sequences=False → 
#  only retun at last (or we can say) 
#    Only the final output of the RNN is passed to the next layer (not the full sequence)
model.add(SimpleRNN(32, input_shape=(50,1), return_sequences=False))

# Add a Dense (fully connected) output layer
# 1 neuron → for binary classification (positive/negative)
# sigmoid → outputs value between 0 and 1 (probability)
model.add(Dense(1, activation='sigmoid'))

# Display model architecture (layers, output shapes, parameters)
model.summary()

C:\Users\p4pri\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 32)                  │           1,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
# Compile model
model.compile(
    loss='binary_crossentropy',   # for binary classification
    optimizer='adam',
    metrics=['accuracy']
)

# Train model
model.fit(
    X_train, y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.5050 - loss: 0.6955 - val_accuracy: 0.5028 - val_loss: 0.6944
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5154 - loss: 0.6929 - val_accuracy: 0.5048 - val_loss: 0.6950
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.5039 - loss: 0.6934 - val_accuracy: 0.5029 - val_loss: 0.6937
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.5092 - loss: 0.6932 - val_accuracy: 0.5059 - val_loss: 0.6948
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5075 - loss: 0.6928 - val_accuracy: 0.5059 - val_loss: 0.6934


# Why Your RNN Had ~50% Accuracy

In [63]:
# The Problem: Raw Integers as Input

# When you use SimpleRNN(32, input_shape=(50,1)),
# you're feeding word IDs like [2071, 56, 26, ...] directly into the RNN.

# The model sees these as raw numbers, so it thinks:
# - 2071 is ~37x larger than 56
# - Therefore word 2071 is "more important" or "greater than" word 56

# But that's completely wrong — word IDs are just arbitrary labels.
# The number 2071 doesn't mean the word is bigger, rarer, or more important than word 56.
# There is no mathematical relationship between them.

# The RNN ends up trying to learn patterns from meaningless numeric comparisons,
# which is why it gets stuck around 50% accuracy (basically random guessing).